# Live/Dead Cell Classification â€” Analysis

## Part 1: Data Exploration

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from PIL import Image

sns.set_theme(style="whitegrid")

DATA_ROOT = Path(os.getenv("DATA_ROOT", ".")).resolve()
df = pd.read_csv(DATA_ROOT / "images" / "metadata.csv")

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"Shape: {df.shape}")
print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
df.head()

### Experiments and assays

The dataset contains **two imaging experiments**. Each experiment has **two assays** (four assays total).

In [ ]:
exp_summary = (
    df.groupby(["exp_id", "assay_id"])
    .size()
    .reset_index(name="n_samples")
    .sort_values(["exp_id", "assay_id"])
)
exp_summary["exp_short"] = exp_summary["exp_id"].str.split("-").str[-1]
exp_summary["assay_short"] = exp_summary["assay_id"].str.split("-").str[-1]

for exp_id, group in exp_summary.groupby("exp_id"):
    exp_short = exp_id.split("-")[-1]
    print(f"Experiment {exp_short} ({exp_id}): {group['n_samples'].sum():,} samples total")
    for _, row in group.iterrows():
        print(f"  assay {row.assay_short}: {row.n_samples:,} samples")
    print()

print(f"Grand total: {len(df):,} samples across {df['exp_id'].nunique()} experiments and {df['assay_id'].nunique()} assays")
display(exp_summary[["exp_short", "assay_short", "n_samples"]])

### Live/dead distribution â€” all samples

Overall class balance across the full dataset.

In [ ]:
counts = df["label"].value_counts()
pcts = df["label"].value_counts(normalize=True) * 100
for label in counts.index:
    print(f"{label}: {counts[label]:,} ({pcts[label]:.1f}%)")

fig, ax = plt.subplots(figsize=(5, 4))
counts.plot(kind="bar", ax=ax, color=["#4CAF50", "#E53935"])
ax.set_xlabel("Label")
ax.set_ylabel("Count")
ax.set_title("Class balance â€” all samples")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for i, v in enumerate(counts.values):
    ax.text(i, v + 100, f"{pcts.iloc[i]:.1f}%", ha="center")
plt.tight_layout()
plt.show()

### Live/dead distribution â€” by experiment and assay

In [ ]:
by_exp = df.groupby(["exp_id", "label"]).size().unstack(fill_value=0)
by_exp["pct_dead"] = (by_exp["dead"] / (by_exp["live"] + by_exp["dead"]) * 100).round(1)
by_exp = by_exp[["live", "dead", "pct_dead"]]
print("Per experiment:")
display(by_exp)

by_assay = df.groupby(["exp_id", "assay_id", "label"]).size().unstack(fill_value=0)
by_assay["pct_dead"] = (by_assay["dead"] / (by_assay["live"] + by_assay["dead"]) * 100).round(1)
by_assay = by_assay[["live", "dead", "pct_dead"]]
print("\nPer assay:")
display(by_assay)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

by_exp[["live", "dead"]].plot(kind="bar", stacked=True, ax=axes[0], color=["#4CAF50", "#E53935"])
axes[0].set_title("By experiment")
axes[0].set_xlabel("")
axes[0].set_ylabel("Count")
axes[0].set_xticklabels([e.split("-")[-1] for e in by_exp.index], rotation=0)
axes[0].legend(title="label")
for i, pct in enumerate(by_exp["pct_dead"]):
    axes[0].text(i, by_exp.loc[by_exp.index[i], ["live", "dead"]].sum() + 80, f"{pct:.1f}% dead", ha="center", fontsize=9)

by_assay[["live", "dead"]].plot(kind="bar", stacked=True, ax=axes[1], color=["#4CAF50", "#E53935"])
axes[1].set_title("By assay")
axes[1].set_xlabel("")
axes[1].set_ylabel("Count")
axes[1].set_xticklabels([f"{e.split('-')[-1]}/{a.split('-')[-1]}" for e, a in by_assay.index], rotation=45, ha="right")
axes[1].legend(title="label")
for i, pct in enumerate(by_assay["pct_dead"]):
    total = by_assay.loc[by_assay.index[i], ["live", "dead"]].sum()
    axes[1].text(i, total + 40, f"{pct:.1f}%", ha="center", fontsize=8)

plt.tight_layout()
plt.show()

### Spatial distribution maps

Each panel shows one assay using the field-of-view coordinates. Live or clean samples are green; dead or noisy samples are red. The uniformity score uses a 4x4 spatial binning of the red points: values near 1 indicate a more uniform spread, while lower values indicate clustering in fewer assay regions.

In [ ]:
def spatial_uniformity_score(x, y, x_range, y_range, bins=4):
    """Normalized entropy of point counts across a fixed 2D assay grid; 1 is uniform, 0 is clustered."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]
    if len(x) == 0:
        return np.nan
    if len(x) == 1 or x_range[0] == x_range[1] or y_range[0] == y_range[1]:
        return 0.0

    counts, _, _ = np.histogram2d(x, y, bins=bins, range=[x_range, y_range])
    probs = counts.ravel() / counts.sum()
    probs = probs[probs > 0]
    return float(-(probs * np.log(probs)).sum() / np.log(bins * bins))


def plot_spatial_assay_maps(data, positive_mask, positive_name, negative_name, title, bins=4):
    plot_df = data.copy()
    plot_df["is_positive"] = positive_mask.to_numpy() if hasattr(positive_mask, "to_numpy") else positive_mask

    assays = (
        plot_df[["exp_id", "assay_id"]]
        .drop_duplicates()
        .sort_values(["exp_id", "assay_id"])
        .itertuples(index=False)
    )
    assays = list(assays)

    fig, axes = plt.subplots(2, 2, figsize=(11, 9), sharex=True, sharey=True)
    rows = []
    colors = {False: "#4CAF50", True: "#E53935"}
    labels = {False: negative_name, True: positive_name}

    for ax, assay in zip(axes.flat, assays):
        subset = plot_df[(plot_df["exp_id"] == assay.exp_id) & (plot_df["assay_id"] == assay.assay_id)]
        for is_positive in [False, True]:
            group = subset[subset["is_positive"] == is_positive]
            ax.scatter(
                group["x_fov"],
                group["y_fov"],
                s=12,
                alpha=0.55,
                c=colors[is_positive],
                label=labels[is_positive],
                edgecolors="none",
            )

        target = subset[subset["is_positive"]]
        x_range = (subset["x_fov"].min(), subset["x_fov"].max())
        y_range = (subset["y_fov"].min(), subset["y_fov"].max())
        score = spatial_uniformity_score(target["x_fov"], target["y_fov"], x_range, y_range, bins=bins)
        rows.append(
            {
                "exp_id": assay.exp_id,
                "assay_id": assay.assay_id,
                f"n_{positive_name}": len(target),
                f"pct_{positive_name}": 100 * len(target) / len(subset),
                f"{positive_name}_uniformity_score": score,
            }
        )

        ax.set_title(
            f"exp {assay.exp_id[-4:]} / assay {assay.assay_id[-4:]}\n"
            f"{positive_name} uniformity={score:.2f}",
            fontsize=10,
        )
        ax.set_xlabel("x_fov")
        ax.set_ylabel("y_fov")
        ax.invert_yaxis()
        ax.legend(loc="best", fontsize=8)

    for ax in axes.flat[len(assays):]:
        ax.axis("off")

    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

    summary = pd.DataFrame(rows)
    display(summary.round({f"pct_{positive_name}": 1, f"{positive_name}_uniformity_score": 3}))
    return summary


dead_spatial_summary = plot_spatial_assay_maps(
    df,
    df["label"].eq("dead"),
    positive_name="dead",
    negative_name="live",
    title="Live/dead spatial distribution by assay",
)

### Image examples

Live vs dead cells from each assay, and a comparison of `r_fov=21` vs `r_fov=36`.

In [ ]:
assays = df["assay_id"].unique()
samples = []
for assay in assays:
    subset = df[df["assay_id"] == assay]
    samples.append(subset[subset["label"] == "live"].sample(2, random_state=42))
    samples.append(subset[subset["label"] == "dead"].sample(2, random_state=42))
gallery = pd.concat(samples)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, (_, row) in zip(axes.flat, gallery.iterrows()):
    img = Image.open(DATA_ROOT / "images" / row.filepath)
    ax.imshow(img, cmap="gray")
    ax.set_title(f"{row.label} | exp {row.exp_id[-4:]} | assay {row.assay_id[-4:]} | r_fov={int(row.r_fov)}", fontsize=9)
    ax.axis("off")
plt.suptitle("2 live + 2 dead per assay", y=1.01)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for col, label in enumerate(["live", "dead"]):
    for row_idx, r_fov in enumerate([21, 36]):
        subset = df[(df["r_fov"] == r_fov) & (df["label"] == label)].sample(3, random_state=42)
        for i, (_, sample) in enumerate(subset.iterrows()):
            ax = axes[row_idx, col * 3 + i]
            img = Image.open(DATA_ROOT / "images" / sample.filepath)
            ax.imshow(img, cmap="gray")
            ax.set_title(f"{label} | r_fov={int(r_fov)}", fontsize=9)
            ax.axis("off")
plt.suptitle("r_fov=21 (top) vs r_fov=36 (bottom)", y=1.02)
plt.tight_layout()
plt.show()

### Image quality

Metrics computed on the full dataset. Outliers (IQR method) are saved to `quality_outliers.csv`.

In [ ]:

BG = 231
METRICS = ["mean_intensity", "std_intensity", "dynamic_range", "pct_zero", "sharpness"]


def image_metrics(filepath):
    img = np.array(Image.open(DATA_ROOT / "images" / filepath), dtype=np.float64)
    well = img[img != BG]
    laplacian = (
        -4 * img[1:-1, 1:-1]
        + img[:-2, 1:-1] + img[2:, 1:-1]
        + img[1:-1, :-2] + img[1:-1, 2:]
    )
    center_well = img[1:-1, 1:-1] != BG
    return {
        "mean_intensity": well.mean(),
        "std_intensity": well.std(),
        "dynamic_range": well.max() - well.min(),
        "pct_zero": (well == 0).mean() * 100,
        "sharpness": laplacian[center_well].var(),
    }


print("Computing quality metrics on full dataset...")
quality = df[["sample_id", "filepath", "label", "exp_id"]].copy()
metrics_df = pd.DataFrame(quality["filepath"].apply(image_metrics).tolist())
quality = pd.concat([quality.reset_index(drop=True), metrics_df], axis=1)

print("Summary statistics:")
display(quality[METRICS].describe().round(2))

fig, axes = plt.subplots(1, len(METRICS), figsize=(18, 3.5))
for ax, metric in zip(axes, METRICS):
    sns.boxplot(data=quality, x="label", y=metric, ax=ax)
    ax.set_title(metric.replace("_", " "))
    ax.set_xlabel("")
plt.suptitle("Image quality by label", y=1.05)
plt.tight_layout()
plt.show()

# --- outlier detection (IQR) ---
def iqr_outlier_mask(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (series < q1 - k * iqr) | (series > q3 + k * iqr)


outlier_flags = {}
for metric in METRICS:
    col = f"outlier_{metric}"
    quality[col] = iqr_outlier_mask(quality[metric])
    outlier_flags[metric] = col

outlier_cols = list(outlier_flags.values())
quality["n_outlier_metrics"] = quality[outlier_cols].sum(axis=1)
quality["outlier_in"] = quality.apply(
    lambda r: ",".join(m for m in METRICS if r[f"outlier_{m}"]), axis=1
)

outliers = quality[quality["n_outlier_metrics"] > 0].copy()
outlier_export_cols = [
    "sample_id", "filepath", "label", "exp_id",
    *METRICS,
    *outlier_cols,
    "n_outlier_metrics", "outlier_in",
]
outliers[outlier_export_cols].to_csv(DATA_ROOT / "quality_outliers.csv", index=False)

print(f"\nOutliers saved to {DATA_ROOT / 'quality_outliers.csv'}")
print(f"Total outlier samples: {len(outliers):,} / {len(quality):,}")
print(f"  single-category: {(outliers['n_outlier_metrics'] == 1).sum():,}")
print(f"  multi-category:  {(outliers['n_outlier_metrics'] > 1).sum():,}")
print("\nOutliers per metric:")
for metric in METRICS:
    print(f"  {metric}: {quality[f'outlier_{metric}'].sum():,}")


### Quality outlier examples

Single-category outliers (flagged in exactly one metric) and multi-category outliers (flagged in two or more).

In [ ]:
single = outliers[outliers["n_outlier_metrics"] == 1]

# --- single-category outliers: 2 examples per metric ---
examples_per_metric = []
for metric in METRICS:
    col = f"outlier_{metric}"
    subset = single[single[col]].head(2)
    for _, row in subset.iterrows():
        examples_per_metric.append((metric, row))

n = len(examples_per_metric)
cols = 4
rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
for ax, (metric, row) in zip(axes.flat, examples_per_metric):
    img = Image.open(DATA_ROOT / "images" / row.filepath)
    ax.imshow(img, cmap="gray")
    val = row[metric]
    ax.set_title(f"{metric}\n{row.sample_id} | {val:.1f}", fontsize=8)
    ax.axis("off")
for ax in axes.flat[n:]:
    ax.axis("off")
plt.suptitle("Single-category outliers (2 per metric)", y=1.01)
plt.tight_layout()
plt.show()

# --- multi-category outliers ---
multi = outliers[outliers["n_outlier_metrics"] > 1].sort_values("n_outlier_metrics", ascending=False).head(9)

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, (_, row) in zip(axes.flat, multi.iterrows()):
    img = Image.open(DATA_ROOT / "images" / row.filepath)
    ax.imshow(img, cmap="gray")
    ax.set_title(f"{row.sample_id}\n{row.n_outlier_metrics} metrics: {row.outlier_in}", fontsize=7)
    ax.axis("off")
for ax in axes.flat[len(multi):]:
    ax.axis("off")
plt.suptitle("Multi-category outliers", y=1.01)
plt.tight_layout()
plt.show()

### Observations

- **Two experiments, four assays**: exp 5738 (1,904 samples, 2 assays) and exp 9996 (8,923 samples, 2 assays).
- **Class imbalance**: ~76% live / ~24% dead overall; models will need to account for this.
- **Dead rate varies by assay**: ranges from ~23% to ~43%, so batch effects matter.
- **Circular mask is consistent**: all images are 224Ã—224; effective radius from pixels â‰  231 matches metadata `r` (~52.27 for r_fov=21, ~89.6 for r_fov=36).
- **r_fov=36**: one assay uses larger wells that fill more of the 224 px crop â€” a potential domain shift.
- **Brightness, contrast, and sharpness differ between experiments**, suggesting batch effects in exposure or focus.
- **Quality outliers**: saved to `quality_outliers.csv`; single-metric flags often reflect brightness or focus extremes, multi-metric flags suggest broadly low-quality acquisitions.
- **Subtle BF signal**: live vs dead is not always obvious by eye; classification will rely on fine-grained texture differences.


---

## Part 2: Data Cleaning and Split

### Data cleaning

Samples flagged as outliers in **3 or more** of the 5 quality metrics are labelled `noisy` and excluded from train/val/test. They are tracked in the split as `noisy` so they can be analysed later.


In [ ]:

NOISY_THRESHOLD = 3

outliers_df = pd.read_csv(DATA_ROOT / "quality_outliers.csv")
noisy_ids = set(outliers_df.loc[outliers_df["n_outlier_metrics"] >= NOISY_THRESHOLD, "sample_id"])

df_clean = df[~df["sample_id"].isin(noisy_ids)].copy()
df_noisy = df[df["sample_id"].isin(noisy_ids)].copy()

print(f"Total samples:  {len(df):,}")
print(f"Noisy (>= {NOISY_THRESHOLD} flags): {len(df_noisy):,} ({100*len(df_noisy)/len(df):.1f}%)")
print(f"Clean:          {len(df_clean):,}")
print(f"\nNoisy by label:\n{df_noisy['label'].value_counts()}")
print(f"\nNoisy by assay:\n{df_noisy.groupby('assay_id').size()}")


### Noisy sample spatial distribution

This repeats the assay maps for the quality split. Clean samples are green and noisy samples are red; the uniformity score measures whether noisy samples are spread across the assay or concentrated in particular field-of-view regions.

In [ ]:
noisy_spatial_summary = plot_spatial_assay_maps(
    df,
    df["sample_id"].isin(noisy_ids),
    positive_name="noisy",
    negative_name="clean",
    title="Noisy/clean spatial distribution by assay",
)

### Train / val / test split

We keep **all** clean samples and handle the class imbalance during training with a **weighted loss** (plus normalization and augmentation). No data is discarded or subset to force balance.

**Strategy:**

- Stratify by `(assay_id, label)` â€” 8 groups â€” and split each group 70 / 15 / 15.
- This preserves the natural class ratio *and* the assay / r_fov distribution identically across train, val, and test.
- `noisy` samples (3 or more outlier flags out of 5 quality checks) get their own split label and are excluded from modelling.

**Why val/test stay imbalanced:**  
Validation and test should mirror the real deployment distribution, so they keep the natural ~79/21 live/dead ratio. Imbalance is corrected only in the *training* loss. We therefore evaluate with imbalance-robust metrics (per-class recall, F1, balanced accuracy, PR-AUC) rather than raw accuracy, and report them **per assay** to surface batch-specific behaviour.

In [ ]:
VAL_FRAC  = 0.15
TEST_FRAC = 0.15
SEED = 42

def assign_splits(sample_ids):
    """Shuffle ids and split into train/val/test by fraction."""
    ids = list(sample_ids)
    rng = np.random.default_rng(SEED)
    rng.shuffle(ids)
    n = len(ids)
    n_test = max(1, round(n * TEST_FRAC))
    n_val = max(1, round(n * VAL_FRAC))
    return {
        "test": ids[:n_test],
        "val": ids[n_test:n_test + n_val],
        "train": ids[n_test + n_val:],
    }

split_map = {}

# stratify by (assay, label): keeps assay/r_fov and natural class ratio identical across splits
for (assay_id, label), group in df_clean.groupby(["assay_id", "label"]):
    for split_name, ids in assign_splits(group["sample_id"]).items():
        for sid in ids:
            split_map[sid] = split_name

for sid in df_noisy["sample_id"]:
    split_map[sid] = "noisy"

split = pd.DataFrame({"sample_id": list(split_map), "split": list(split_map.values())})
split.to_csv(DATA_ROOT / "split.csv", index=False)
print(f"split.csv written â€” {len(split):,} rows")
print(f"\nSplit counts:\n{split['split'].value_counts()}")

### Split validation

Verify assay balance, class balance, and no leakage across splits.

In [ ]:
split_meta = split.merge(df[["sample_id", "label", "assay_id", "exp_id", "r_fov"]], on="sample_id")

# no sample appears in more than one split
assert split_meta["sample_id"].nunique() == len(split_meta), "Duplicate sample_ids!"
# every row in df is accounted for
assert len(split_meta) == len(df), f"Missing rows: {len(df) - len(split_meta)}"
print("No leakage: all sample_ids unique and all rows accounted for.")

core_splits = ["train", "val", "test"]
core = split_meta[split_meta["split"].isin(core_splits)]

print("\nLabel ratio per split (natural imbalance preserved):")
label_ratio = core.groupby(["split", "label"]).size().unstack(fill_value=0)
label_ratio["pct_dead"] = (label_ratio["dead"] / label_ratio.sum(axis=1) * 100).round(1)
display(label_ratio)

print("\nAssay distribution across core splits:")
display(core.groupby(["split", "assay_id"]).size().unstack(fill_value=0))

print("\nr_fov distribution across core splits:")
display(core.groupby(["split", "r_fov"]).size().unstack(fill_value=0))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = {"train": "#4CAF50", "val": "#FFC107", "test": "#2196F3", "noisy": "#E53935"}
split_counts = split_meta["split"].value_counts().reindex(["train", "val", "test", "noisy"])
split_counts.plot(kind="bar", ax=axes[0], color=[colors[s] for s in split_counts.index])
axes[0].set_title("Overall split sizes")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].set_ylabel("Count")

label_by_split = core.groupby(["split", "label"]).size().unstack(fill_value=0)
label_by_split.plot(kind="bar", stacked=True, ax=axes[1], color=["#E53935", "#4CAF50"])
axes[1].set_title("Label counts (core splits)")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

assay_by_split = core.groupby(["split", "assay_id"]).size().unstack(fill_value=0)
assay_by_split.columns = [c.split("-")[-1] for c in assay_by_split.columns]
assay_by_split.plot(kind="bar", stacked=True, ax=axes[2])
axes[2].set_title("Assay distribution (core splits)")
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

# --- training class weights (global inverse frequency) ---
train_labels = core[core["split"] == "train"]["label"]
n_train = len(train_labels)
class_weights = {
    label: n_train / (2 * count)
    for label, count in train_labels.value_counts().items()
}
print(f"\nGlobal class weights for weighted loss (from train split):")
for label, w in class_weights.items():
    print(f"  {label}: {w:.3f}")

### Summary of findings

The split keeps the assignment's leakage risks under control by assigning each `sample_id` exactly once and stratifying clean samples by `(assay_id, label)`. This preserves class balance and assay representation across train, validation, and test while excluding only the samples that fail the stricter quality rule of 3 or more failed checks out of 5.

The `noisy` split is retained rather than discarded entirely, so low-quality acquisitions can still be analysed separately. Validation and test remain naturally imbalanced to reflect deployment conditions; class imbalance is handled only in the training loss.

---

## Part 3: Model Training and Evaluation

**Model:** ResNet18 (ImageNet pretrained) with a 2-class head.

**Why ResNet18:** ResNet18 is a conservative baseline for this assignment: it is strong enough to learn subtle brightfield texture and morphology cues, but small enough to train quickly and inspect failure modes without spending the whole effort on architecture tuning. ImageNet pretraining gives useful low-level edge/texture filters even though the microscopy images are grayscale; the input is replicated to 3 channels to reuse those pretrained weights.

**Why weighted cross-entropy loss:** The dataset is naturally imbalanced, with many more live than dead cells. Weighted cross-entropy keeps all clean samples in training while increasing the penalty for errors on the minority class, which is important because dead-cell recall is a core objective and plain accuracy would be misleading.

**Why an L4/L40S-class GPU:** The model and 224x224 images are modest, so a single cloud GPU is sufficient. The GPU accelerates the repeated CNN forward/backward passes, enables larger batches, and makes the notebook reproducible within a practical runtime while still leaving the analysis focused on data quality and evaluation rather than infrastructure.

**Why batch size 512:** Earlier monitoring showed the GPU was under-fed at smaller batch sizes. Batch size 512 uses more available GPU memory and raises utilization close to saturation, improving throughput while still fitting comfortably in VRAM. Validation macro F1 remains the checkpoint selection criterion, so the larger batch is accepted only if generalization remains healthy.

**Training choices:**
- Global weighted cross-entropy to handle class imbalance (live ~76%, dead ~24%)
- Grayscale replicated to 3 channels for transfer learning
- Train-set normalization (accounts for batch brightness differences)
- Augmentation: flips, rotation, brightness/contrast jitter
- 30 epochs, batch size 512, 12 data-loader workers on the cloud GPU
- Best checkpoint selected by validation macro F1

In [ ]:
import os
import subprocess
import sys


def artifact_path(name):
    for folder in ("checkpoints", "checkpoints_from_nebius"):
        path = Path(folder) / name
        if path.exists():
            return path
    return Path("checkpoints") / name


def run_script(args, log_name=None):
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    cmd = [sys.executable, "-u", *args]
    log_path = Path("checkpoints") / log_name if log_name else None
    if log_path:
        log_path.parent.mkdir(parents=True, exist_ok=True)

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    proc.wait()
    if log_path:
        log_path.write_text("".join(lines))
        print(f"\nLog saved to {log_path}")
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}): {' '.join(cmd)}")


run_script(
    ["train.py", "--epochs", "30", "--batch-size", "512", "--num-workers", "12", "--output-dir", "checkpoints"],
    log_name="train.log",
)

In [ ]:
import json

history = json.loads(artifact_path("history.json").read_text())
hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

axes[0].plot(hist_df["epoch"], hist_df["train_loss"], label="train")
axes[0].plot(hist_df["epoch"], hist_df["val_loss"], label="val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training and validation loss")
axes[0].legend()

axes[1].plot(hist_df["epoch"], hist_df["val_f1"], label="val F1")
axes[1].plot(hist_df["epoch"], hist_df["val_balanced_acc"], label="val balanced acc")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_title("Validation metrics")
axes[1].legend()

plt.tight_layout()
plt.show()
display(hist_df.tail())

In [ ]:
run_script(
    ["evaluate.py", "--checkpoint", str(artifact_path("best.pt")), "--split", "test"],
    log_name="eval.log",
)

In [ ]:
import torch
from sklearn.metrics import average_precision_score, precision_recall_curve
from torch.utils.data import DataLoader
from train import build_model, CellDataset, get_transforms, load_dataframe

results = json.loads(artifact_path("eval_results.json").read_text())
cm = np.array(results["overall"]["confusion_matrix"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(artifact_path("best.pt"), map_location=device, weights_only=False)
test_df = load_dataframe(DATA_ROOT)
test_df = test_df[test_df["split"] == "test"].reset_index(drop=True)

dataset = CellDataset(
    test_df,
    DATA_ROOT,
    get_transforms(checkpoint["normalize_mean"], checkpoint["normalize_std"]),
)
loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)

model = build_model().to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

y_true, y_prob_dead = [], []
with torch.no_grad():
    for x, y in loader:
        logits = model(x.to(device))
        prob_dead = torch.softmax(logits, dim=1)[:, 1]
        y_true.extend(y.tolist())
        y_prob_dead.extend(prob_dead.cpu().tolist())

y_true = np.array(y_true)
y_prob_dead = np.array(y_prob_dead)
ap = average_precision_score(y_true, y_prob_dead)
precision, recall, thresholds = precision_recall_curve(y_true, y_prob_dead)
baseline = (y_true == 1).mean()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["pred live", "pred dead"],
    yticklabels=["true live", "true dead"],
    ax=axes[0],
)
axes[0].set_title("Test set confusion matrix")

axes[1].plot(recall, precision, label=f"dead class (AP={ap:.3f})")
axes[1].hlines(baseline, 0, 1, linestyles="--", colors="gray", label=f"prevalence={baseline:.3f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Test set precision-recall curve")
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Average precision (dead): {ap:.4f}")

In [ ]:
metric_rows = [
    ("f1_macro", "F1 (macro)"),
    ("balanced_accuracy", "Balanced accuracy"),
    ("precision_live", "Precision (live)"),
    ("recall_live", "Recall (live)"),
    ("precision_dead", "Precision (dead)"),
    ("recall_dead", "Recall (dead)"),
]

exp_ids = sorted(results["by_exp_id"])
assay_ids = sorted(results["by_assay_id"])

column_metrics = {"Overall": results["overall"]}
for i, exp_id in enumerate(exp_ids, start=1):
    column_metrics[f"exp {i}"] = results["by_exp_id"][exp_id]
for i, assay_id in enumerate(assay_ids, start=1):
    column_metrics[f"assay {i}"] = results["by_assay_id"][assay_id]

eval_table = pd.DataFrame(
    {
        col: {label: metrics[key] for key, label in metric_rows}
        for col, metrics in column_metrics.items()
    }
)
eval_table.index.name = "metric"

display(eval_table.round(3))

print("Column mapping:")
for i, exp_id in enumerate(exp_ids, start=1):
    print(f"  exp {i}: {exp_id}")
for i, assay_id in enumerate(assay_ids, start=1):
    print(f"  assay {i}: {assay_id}")

In [ ]:
import time

import torch
from torch.utils.data import DataLoader
from train import build_model, CellDataset, get_transforms, load_dataframe

# Inference throughput benchmark on the test split.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(artifact_path("best.pt"), map_location=device, weights_only=False)
test_df = load_dataframe(DATA_ROOT)
test_df = test_df[test_df["split"] == "test"].reset_index(drop=True)
batch_size = 64

dataset = CellDataset(
    test_df,
    DATA_ROOT,
    get_transforms(checkpoint["normalize_mean"], checkpoint["normalize_std"]),
)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=device.type == "cuda")

model = build_model().to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

with torch.no_grad():
    for i, (x, _) in enumerate(loader):
        model(x.to(device))
        if i >= 4:
            break

n_samples = 0
t0 = time.perf_counter()
with torch.no_grad():
    for x, _ in loader:
        model(x.to(device))
        n_samples += len(x)
elapsed = time.perf_counter() - t0
throughput = n_samples / elapsed

print(f"Device: {device}")
print(f"Batch size: {batch_size}")
print(f"Throughput: {throughput:.1f} samples/sec ({n_samples:,} samples in {elapsed:.2f}s)")

# GPU utilization and memory were recorded once per training epoch by train.py.
has_gpu_stats = {"gpu_util_pct", "gpu_mem_used_mb"}.issubset(hist_df.columns)
if has_gpu_stats:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(hist_df["epoch"], hist_df["gpu_util_pct"], marker="o")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Utilization (%)")
    axes[0].set_title("GPU utilization during training")

    axes[1].plot(hist_df["epoch"], hist_df["gpu_mem_used_mb"], marker="o", color="#7E57C2")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Memory used (MB)")
    axes[1].set_title("GPU memory usage during training")

    plt.tight_layout()
    plt.show()

    perf_summary = pd.DataFrame(
        {
            "metric": [
                "inference_throughput_samples_per_sec",
                "mean_gpu_util_pct",
                "max_gpu_mem_used_mb",
            ],
            "value": [
                throughput,
                hist_df["gpu_util_pct"].mean(),
                hist_df["gpu_mem_used_mb"].max(),
            ],
        }
    )
else:
    print("GPU stats were not found in history.json; rerun training with the updated train.py to collect them.")
    perf_summary = pd.DataFrame(
        {"metric": ["inference_throughput_samples_per_sec"], "value": [throughput]}
    )

display(perf_summary.round(2))

### Summary of results

The latest test evaluation shows strong overall performance, with macro F1 around 0.805 and balanced accuracy around 0.845. The confusion matrix highlights the main tradeoff: dead-cell recall is high, while dead-cell precision is lower because some live cells are predicted as dead.

Performance is not uniform across experiments and assays, so the per-experiment and per-assay table should be used when interpreting batch effects. The precision-recall curve summarizes this same tradeoff for the dead class, and the final performance cell reports inference throughput together with GPU utilization and memory usage from training.


---

## Part 4: Error Analysis

Compare **misclassified** vs **correctly classified** test samples. Each example is annotated with quality-outlier flags from Part 1 (`quality_outliers.csv`). Samples with **>=3 outlier metrics** were labelled `noisy` in the split and excluded from train/val/test; test samples may still carry 1-2 flags.


In [ ]:
import torch
from torch.utils.data import DataLoader
from train import IDX_TO_LABEL, build_model, CellDataset, get_transforms, load_dataframe

NOISY_THRESHOLD = 3
checkpoint_path = Path("checkpoints/best.pt")
if not checkpoint_path.exists():
    checkpoint_path = Path("checkpoints_from_nebius/best.pt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
mean, std = checkpoint["normalize_mean"], checkpoint["normalize_std"]

test_df = load_dataframe(DATA_ROOT)
test_df = test_df[test_df["split"] == "test"].reset_index(drop=True)

dataset = CellDataset(test_df, DATA_ROOT, get_transforms(mean, std, train=False))
loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)

model = build_model().to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

y_true, y_pred, y_prob = [], [], []
with torch.no_grad():
    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        prob = torch.softmax(logits, dim=1)
        y_true.extend(y.tolist())
        y_pred.extend(logits.argmax(dim=1).cpu().tolist())
        y_prob.extend(prob.cpu().tolist())

pred_df = test_df.copy()
pred_df["true_idx"] = y_true
pred_df["pred_idx"] = y_pred
pred_df["true_label"] = pred_df["true_idx"].map(IDX_TO_LABEL)
pred_df["pred_label"] = pred_df["pred_idx"].map(IDX_TO_LABEL)
pred_df["prob_live"] = [p[0] for p in y_prob]
pred_df["prob_dead"] = [p[1] for p in y_prob]
pred_df["correct"] = pred_df["true_label"] == pred_df["pred_label"]

quality_flags = pd.read_csv(DATA_ROOT / "quality_outliers.csv")
flag_cols = [c for c in quality_flags.columns if c.startswith("outlier_") and c != "outlier_in"]
pred_df = pred_df.merge(
    quality_flags[["sample_id", "n_outlier_metrics", "outlier_in"] + flag_cols],
    on="sample_id",
    how="left",
)
pred_df["n_outlier_metrics"] = pred_df["n_outlier_metrics"].fillna(0).astype(int)
pred_df["outlier_in"] = pred_df["outlier_in"].fillna("none")
for col in flag_cols:
    pred_df[col] = pred_df[col].fillna(False)

pred_df["noisy_split"] = pred_df["split"] == "noisy"
pred_df["noisy_threshold"] = pred_df["n_outlier_metrics"] >= NOISY_THRESHOLD

misclassified = pred_df[~pred_df["correct"]].copy()
correct = pred_df[pred_df["correct"]].copy()

print(f"Test set: {len(pred_df):,} samples")
print(f"  misclassified: {len(misclassified):,}")
print(f"  with any outlier flag: {(pred_df['n_outlier_metrics'] > 0).sum():,}")
print(f"  with â‰¥{NOISY_THRESHOLD} flags (noisy threshold): {pred_df['noisy_threshold'].sum():,}")


def example_title(row):
    noisy_tag = ""
    if row["noisy_split"]:
        noisy_tag = " | NOISY split"
    elif row["noisy_threshold"]:
        noisy_tag = f" | â‰¥{NOISY_THRESHOLD} flags"
    flags = "none" if row["outlier_in"] == "none" else row["outlier_in"]
    return (
        f"true={row['true_label']} â†’ pred={row['pred_label']}{noisy_tag}\n"
        f"p(dead)={row['prob_dead']:.2f} | {int(row['n_outlier_metrics'])} flags: {flags}"
    )


def pick_examples(df, n, label=None, correct=None, prefer_outliers=False):
    subset = df.copy()
    if label is not None:
        subset = subset[subset["true_label"] == label]
    if correct is not None:
        subset = subset[subset["correct"] == correct]
    if len(subset) == 0:
        return subset
    if prefer_outliers:
        subset = subset.sort_values(["n_outlier_metrics", "prob_dead"], ascending=[False, True])
    else:
        subset = subset.sort_values("prob_dead", ascending=(label == "live"))
    return subset.drop_duplicates("assay_id").head(n)


def show_examples(examples, title, ncol=4):
    if examples.empty:
        print(f"No examples for: {title}")
        return
    nrow = int(np.ceil(len(examples) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.2 * nrow))
    axes = np.atleast_1d(axes).flatten()
    for ax, (_, row) in zip(axes, examples.iterrows()):
        img = Image.open(DATA_ROOT / "images" / row.filepath)
        ax.imshow(img, cmap="gray")
        ax.set_title(example_title(row), fontsize=7)
        ax.axis("off")
    for ax in axes[len(examples):]:
        ax.axis("off")
    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()


show_examples(
    pick_examples(misclassified, n=4, label="live", correct=False, prefer_outliers=True),
    "Misclassified â€” true live, predicted dead",
)
show_examples(
    pick_examples(misclassified, n=4, label="dead", correct=False, prefer_outliers=True),
    "Misclassified â€” true dead, predicted live",
)
show_examples(
    pick_examples(correct[correct["n_outlier_metrics"] > 0], n=4, prefer_outliers=True),
    "Correctly classified â€” samples with quality outlier flags",
)
show_examples(
    pick_examples(correct[correct["n_outlier_metrics"] == 0], n=4),
    "Correctly classified â€” no quality outlier flags",
)


In [ ]:
def flag_group(n):
    if n == 0:
        return "0 flags"
    if n == 1:
        return "1 flag"
    if n < NOISY_THRESHOLD:
        return "2 flags"
    return f"â‰¥{NOISY_THRESHOLD} flags"


error_by_flags = pred_df.assign(
    outlier_group=pred_df["n_outlier_metrics"].map(flag_group)
).groupby("outlier_group", observed=True).agg(
    n=("sample_id", "count"),
    error_rate=("correct", lambda s: 1 - s.mean()),
)
display(error_by_flags.round(3))


### Summary of findings and next steps

The error analysis compares misclassified and correctly classified test examples with the quality flags attached, which helps separate likely model mistakes from difficult or low-quality acquisitions. The main observed failure mode is confusion near the live/dead visual boundary: brightfield morphology can be subtle, so the model sometimes predicts dead for live cells that look degraded or predicts live for dead cells that retain live-like texture.

Quality flags are useful but not a complete explanation for errors. Some flagged samples are still classified correctly, and some mistakes occur on otherwise clean images, suggesting that production robustness should combine better data quality control with model and evaluation improvements.

With more time, the top improvements would be: collect or label more examples from the weaker experiments/assays; add stronger assay-aware validation and calibration so confidence scores are reliable across batches; and train a small ensemble or compare stronger pretrained backbones while keeping the same leakage-safe split and per-experiment reporting.